In [1]:
# %% 셀 1: 데이터 로드 (텔롭 bbox 예측 모델)
import os, json, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import polars as pl
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

POS_DIR = "./data/8_telop_position"
STT_DIR = "./data/4_stt_results"
EMB_PATH = "./data/8_text_embeddings.pt"
HEATMAP_DIR = "./data/8_heatmap"
GRID_W = 80
GRID_H = 80
EVAL_PER_CHANNEL = 5
SEED = 42
NUM_WORKERS = 32
FPS = 10
MAX_FRAMES = 2000
MAX_ACTIVE_PER_FRAME = 150
MAX_TEXT_LEN = 200
SLOT_DIM = 5
TIME_DIM = 3

text2emb = torch.load(EMB_PATH, weights_only=True)
EMB_DIM = next(iter(text2emb.values())).shape[0]
ZERO_EMB = torch.zeros(EMB_DIM)
print(f"✅ 임베딩 로드: {len(text2emb):,}개  |  dim={EMB_DIM}")


def stt_frames_to_segments(df_stt):
    rows = df_stt.sort("frame_num").to_dicts()
    segments = []
    cur_text = None
    cur_start_frame = None
    prev_frame = None
    for r in rows:
        t = r["stt_text"]
        f = int(r["frame_num"])
        if t != cur_text:
            if cur_text is not None and cur_text != "":
                segments.append({
                    "start": cur_start_frame / FPS,
                    "end": (prev_frame + 1) / FPS,
                    "text": cur_text.strip(),
                })
            cur_text = t
            cur_start_frame = f
        prev_frame = f
    if cur_text is not None and cur_text != "":
        segments.append({
            "start": cur_start_frame / FPS,
            "end": (prev_frame + 1) / FPS,
            "text": cur_text.strip(),
        })
    return segments


def load_one(args):
    channel, path = args
    with open(path, "r") as f:
        data = json.load(f)
    instances = data.get("instances", [])
    duration = data.get("duration", 0.1)
    if instances:
        duration = max(duration, max(inst["end_sec"] for inst in instances))
    video_name = data.get("video", "")
    file_id = os.path.basename(path)[:-5]
    inst_list = []
    for inst in instances:
        gx = int(np.clip(round(inst["grid_x"]), 0, GRID_W - 1))
        gy = int(np.clip(round(inst["grid_y"]), 0, GRID_H - 1))
        gw = int(np.clip(round(inst["grid_w"]), 1, GRID_W))
        gh = int(np.clip(round(inst["grid_h"]), 1, GRID_H))
        inst_list.append({
            "text": inst["text"],
            "text_len": len(inst["text"]),
            "start": inst["start_sec"],
            "end": inst["end_sec"],
            "x": gx, "y": gy, "w": gw, "h": gh,
        })
    stt_path = os.path.join(STT_DIR, channel, file_id + ".parquet")
    stt_segments = []
    if os.path.exists(stt_path):
        try:
            df_stt = pl.read_parquet(stt_path, glob=False)
            stt_segments = stt_frames_to_segments(df_stt)
        except:
            pass
    return {
        "channel": channel,
        "video_name": video_name,
        "file_id": file_id,
        "instances": inst_list,
        "stt_segments": stt_segments,
        "duration": duration,
    }


json_paths = []
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in sorted(os.listdir(ch_dir)):
        if fname.endswith(".json"):
            json_paths.append((channel, os.path.join(ch_dir, fname)))

samples = []
channel_set = set()
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(load_one, args): args for args in json_paths}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="로드"):
        result = fut.result()
        channel_set.add(result["channel"])
        samples.append(result)

channels = sorted(channel_set)
channel2id = {ch: i for i, ch in enumerate(channels)}

rng = random.Random(SEED)
by_channel = {}
for s in samples:
    if s["channel"] not in by_channel:
        by_channel[s["channel"]] = []
    by_channel[s["channel"]].append(s)

train_samples = []
eval_samples = []
for ch, ch_samples in by_channel.items():
    ch_samples.sort(key=lambda s: s["file_id"])
    rng.shuffle(ch_samples)
    n_eval = min(EVAL_PER_CHANNEL, len(ch_samples))
    eval_samples.extend(ch_samples[:n_eval])
    train_samples.extend(ch_samples[n_eval:])

train_samples = [s for s in train_samples if len(s["instances"]) > 0]
eval_samples_with = [s for s in eval_samples if len(s["instances"]) > 0]

print(f"✅ 채널: {len(channels)}개")
print(f"   train: {len(train_samples):,}  eval: {len(eval_samples_with):,}")

✅ 임베딩 로드: 2,167,019개  |  dim=1024


로드: 100%|██████████| 66400/66400 [00:10<00:00, 6554.20it/s]


✅ 채널: 664개
   train: 56,892  eval: 2,984


In [2]:
rng_ch = random.Random(42)
sampled_channels = rng_ch.sample(channels, 67)
sampled_set = set(sampled_channels)

train_samples = [s for s in train_samples if s["channel"] in sampled_set]
eval_samples = [s for s in eval_samples if s["channel"] in sampled_set]
channels = sampled_channels
channel2id = {ch: i for i, ch in enumerate(channels)}

eval_samples_with = [s for s in eval_samples if len(s["instances"]) > 0]
train_samples = [s for s in train_samples if len(s["instances"]) > 0]

print(f"\n🔬 샘플링: {len(channels)}개 채널")
print(f"   train: {len(train_samples):,}  |  eval: {len(eval_samples):,}")


🔬 샘플링: 67개 채널
   train: 5,574  |  eval: 335


In [3]:
# %% 셀 2: Dataset + DataLoader (slot-conditioned, bbox 기반 80×80)
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

BATCH_SIZE = 8
NUM_WORKERS_DL = 8
MAX_FRAMES = 2000
MAX_ACTIVE_PER_FRAME = 150
MAX_TEXT_LEN = 200
SLOT_DIM = 5  # text_len, ch_sim, vid_sim, stt_sim, has_stt
TIME_DIM = 3  # t_norm, sin(2πt), cos(2πt)
PATCH_SIZE = 8
N_PATCHES_X = GRID_W // PATCH_SIZE  # 10
N_PATCHES_Y = GRID_H // PATCH_SIZE  # 10
N_PATCHES = N_PATCHES_X * N_PATCHES_Y  # 100
POS_WEIGHT = 73.5


class FrameMaskViTDataset(Dataset):
    def __init__(self, samples, channel2id, text2emb):
        self.samples = [s for s in samples if len(s["instances"]) > 0]
        self.channel2id = channel2id
        self.text2emb = text2emb

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        channel_id = self.channel2id[s["channel"]]
        instances = s["instances"]
        duration = max(s["duration"], 0.1)

        n_frames = max(1, min(int(duration * FPS) + 1, MAX_FRAMES))

        # ── 인스턴스 배열 ──
        n_inst = len(instances)
        inst_starts = np.array([inst["start"] for inst in instances])
        inst_ends   = np.array([inst["end"]   for inst in instances])
        inst_tlens  = np.array([inst["text_len"] for inst in instances])
        inst_cx = np.array([inst["x"] for inst in instances])
        inst_cy = np.array([inst["y"] for inst in instances])
        inst_w  = np.array([inst["w"] for inst in instances])
        inst_h  = np.array([inst["h"] for inst in instances])

        # ── 인스턴스별 bbox (x0, y0, x1, y1) 사전 계산 ──
        inst_x0 = np.maximum(0, inst_cx - inst_w // 2)
        inst_y0 = np.maximum(0, inst_cy - inst_h // 2)
        inst_x1 = np.minimum(GRID_W, inst_x0 + inst_w)
        inst_y1 = np.minimum(GRID_H, inst_y0 + inst_h)

        # ── 인스턴스별 유사도 사전 계산 ──
        channel_emb = self.text2emb.get(s["channel"], ZERO_EMB)
        video_emb   = self.text2emb.get(s["video_name"], ZERO_EMB)

        inst_embs = torch.stack([
            self.text2emb.get(inst["text"], ZERO_EMB) for inst in instances
        ])
        ch_sims  = F.cosine_similarity(inst_embs, channel_emb.unsqueeze(0), dim=-1).numpy()
        vid_sims = F.cosine_similarity(inst_embs, video_emb.unsqueeze(0), dim=-1).numpy()

        stt_sims = np.zeros(n_inst, dtype=np.float32)
        has_stts = np.zeros(n_inst, dtype=np.float32)

        stt_segments = s["stt_segments"]
        if len(stt_segments) > 0:
            stt_starts = np.array([seg["start"] for seg in stt_segments])
            stt_ends   = np.array([seg["end"]   for seg in stt_segments])
            stt_embs   = torch.stack([
                self.text2emb.get(seg["text"], ZERO_EMB) for seg in stt_segments
            ])

            for i in range(n_inst):
                mid = (inst_starts[i] + inst_ends[i]) / 2
                stt_active = (stt_starts <= mid) & (stt_ends > mid)
                stt_active_idx = np.where(stt_active)[0]
                if len(stt_active_idx) > 0:
                    stt_sims[i] = F.cosine_similarity(
                        inst_embs[i].unsqueeze(0),
                        stt_embs[stt_active_idx[0]].unsqueeze(0),
                    ).item()
                    has_stts[i] = 1.0

        # ── 프레임별 활성 텔롭 매트릭스 ──
        times = np.arange(n_frames, dtype=np.float32) / FPS
        active_matrix = (
            (inst_starts[None, :] <= times[:, None] + 0.05) &
            (inst_ends[None, :]   >  times[:, None])
        )

        # ── 프레임별 텔롭 슬롯 + bbox ──
        telop_feats = np.zeros((n_frames, MAX_ACTIVE_PER_FRAME, SLOT_DIM), dtype=np.float32)
        telop_mask  = np.zeros((n_frames, MAX_ACTIVE_PER_FRAME), dtype=np.bool_)
        slot_bbox   = np.zeros((n_frames, MAX_ACTIVE_PER_FRAME, 4), dtype=np.int16)

        for fi in range(n_frames):
            active_idx = np.where(active_matrix[fi])[0]
            if len(active_idx) > 0:
                sorted_order = np.argsort(inst_tlens[active_idx])[::-1][:MAX_ACTIVE_PER_FRAME]
                sorted_idx = active_idx[sorted_order]

                for si, ai in enumerate(sorted_idx):
                    telop_feats[fi, si, 0] = inst_tlens[ai] / MAX_TEXT_LEN
                    telop_feats[fi, si, 1] = ch_sims[ai]
                    telop_feats[fi, si, 2] = vid_sims[ai]
                    telop_feats[fi, si, 3] = stt_sims[ai]
                    telop_feats[fi, si, 4] = has_stts[ai]
                    telop_mask[fi, si] = True
                    slot_bbox[fi, si] = [inst_x0[ai], inst_y0[ai],
                                         inst_x1[ai], inst_y1[ai]]

        # ── 시간 feature (3d) ──
        time_feats = np.zeros((n_frames, TIME_DIM), dtype=np.float32)
        t_norm = times / max(duration, 1.0)
        time_feats[:, 0] = t_norm
        time_feats[:, 1] = np.sin(2 * np.pi * t_norm)
        time_feats[:, 2] = np.cos(2 * np.pi * t_norm)

        return {
            "channel_id":  torch.tensor(channel_id, dtype=torch.long),
            "telop_feats": torch.from_numpy(telop_feats),
            "telop_mask":  torch.from_numpy(telop_mask),
            "time_feats":  torch.from_numpy(time_feats),
            "slot_bbox":   torch.from_numpy(slot_bbox.astype(np.int32)),
            "n_frames":    n_frames,
        }


def collate_fn(batch):
    max_T = max(b["n_frames"] for b in batch)
    B = len(batch)

    channel_ids = torch.stack([b["channel_id"] for b in batch])
    telop_feats = torch.zeros(B, max_T, MAX_ACTIVE_PER_FRAME, SLOT_DIM)
    telop_mask  = torch.zeros(B, max_T, MAX_ACTIVE_PER_FRAME, dtype=torch.bool)
    time_feats  = torch.zeros(B, max_T, TIME_DIM)
    slot_bbox   = torch.zeros(B, max_T, MAX_ACTIVE_PER_FRAME, 4, dtype=torch.int32)
    frame_mask  = torch.zeros(B, max_T, dtype=torch.bool)

    for i, b in enumerate(batch):
        T = b["n_frames"]
        telop_feats[i, :T] = b["telop_feats"]
        telop_mask[i, :T]  = b["telop_mask"]
        time_feats[i, :T]  = b["time_feats"]
        slot_bbox[i, :T]   = b["slot_bbox"]
        frame_mask[i, :T]  = True

    return {
        "channel_ids": channel_ids,
        "telop_feats": telop_feats,
        "telop_mask":  telop_mask,
        "time_feats":  time_feats,
        "slot_bbox":   slot_bbox,
        "frame_mask":  frame_mask,
    }


train_ds = FrameMaskViTDataset(train_samples, channel2id, text2emb)
eval_ds  = FrameMaskViTDataset(eval_samples, channel2id, text2emb)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS_DL, pin_memory=True,
    collate_fn=collate_fn, drop_last=True,
    persistent_workers=True,
)
eval_loader = DataLoader(
    eval_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS_DL, pin_memory=True,
    collate_fn=collate_fn,
    persistent_workers=True,
)

print(f"✅ Dataset: train={len(train_ds):,}  eval={len(eval_ds):,}")
print(f"   SLOT_DIM={SLOT_DIM}  MAX_ACTIVE_PER_FRAME={MAX_ACTIVE_PER_FRAME}")
print(f"   N_PATCHES={N_PATCHES}  PATCH_SIZE={PATCH_SIZE}")
print(f"   MAX_FRAMES={MAX_FRAMES}  POS_WEIGHT={POS_WEIGHT}")
print(f"   출력 해상도: {GRID_H}×{GRID_W} (rank-1 분해)")

batch = next(iter(train_loader))
print(f"\n✅ 배치 확인")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")

✅ Dataset: train=5,574  eval=288
   SLOT_DIM=5  MAX_ACTIVE_PER_FRAME=150
   N_PATCHES=100  PATCH_SIZE=8
   MAX_FRAMES=2000  POS_WEIGHT=73.5
   출력 해상도: 80×80 (rank-1 분해)

✅ 배치 확인
  channel_ids: torch.Size([8])
  telop_feats: torch.Size([8, 601, 150, 5])
  telop_mask: torch.Size([8, 601, 150])
  time_feats: torch.Size([8, 601, 3])
  slot_bbox: torch.Size([8, 601, 150, 4])
  frame_mask: torch.Size([8, 601])


In [4]:
# %% 셀 3: 모델 정의 (IntraFrame slot self-attn 2L + DETR decoder 6L)
from torch.utils.checkpoint import checkpoint as ckpt_fn

D_MODEL = 256
N_HEADS = 8
N_LAYERS_TEMPORAL = 4
N_LAYERS_SPATIAL = 4
N_LAYERS_DECODER = 6
N_LAYERS_INTRA_SLOT = 2
D_FF = 512
DROPOUT = 0.1
INTRA_CHUNK = 512
SPATIAL_CHUNK = 512
SLOT_DECODE_CHUNK = 512


class IntraFrameModule(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, d_ff=D_FF,
                 dropout=DROPOUT, n_layers_slot=N_LAYERS_INTRA_SLOT):
        super().__init__()
        self.slot_proj = nn.Sequential(
            nn.Linear(SLOT_DIM, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

        self.slot_pos_emb = nn.Embedding(MAX_ACTIVE_PER_FRAME, d_model)
        nn.init.normal_(self.slot_pos_emb.weight, mean=0.0, std=0.02)

        slot_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu",
            norm_first=True,
        )
        self.slot_self_attn = nn.TransformerEncoder(
            slot_layer, num_layers=n_layers_slot, enable_nested_tensor=True,
        )

        self.pool_query = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.combine = nn.Sequential(
            nn.Linear(d_model * 3 + 1, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, slot_feats, slot_mask):
        N, K, _ = slot_feats.shape
        device = slot_feats.device
        dtype = slot_feats.dtype

        tokens = self.slot_proj(slot_feats)
        d = tokens.shape[-1]

        slot_idx = torch.arange(K, device=device)
        tokens = tokens + self.slot_pos_emb(slot_idx).unsqueeze(0)

        # slot self-attention
        has_any = slot_mask.any(dim=1)
        if has_any.any():
            valid_idx = has_any.nonzero(as_tuple=True)[0]
            valid_tokens = tokens[valid_idx]
            valid_pad = ~slot_mask[valid_idx]

            valid_tokens = self.slot_self_attn(
                valid_tokens, src_key_padding_mask=valid_pad,
            )
            tokens[valid_idx] = valid_tokens.to(tokens.dtype)

        slot_tokens = tokens * slot_mask.unsqueeze(-1).float()

        # pooling
        pooled = torch.zeros(N, d, device=device, dtype=dtype)

        if has_any.any():
            valid_idx = has_any.nonzero(as_tuple=True)[0]
            valid_tokens = tokens[valid_idx]
            valid_mask = slot_mask[valid_idx]
            valid_pad = ~valid_mask
            V = valid_tokens.shape[0]

            count = valid_mask.sum(dim=1, keepdim=True).float()
            count_norm = count / MAX_ACTIVE_PER_FRAME

            masked_tokens = valid_tokens.masked_fill(valid_pad.unsqueeze(-1), 0.0)
            mean_pool = masked_tokens.sum(dim=1) / count.clamp(min=1)

            masked_for_max = valid_tokens.masked_fill(valid_pad.unsqueeze(-1), float("-inf"))
            max_pool = masked_for_max.max(dim=1).values

            q = self.pool_query.expand(V, -1, -1)
            a = torch.bmm(q, valid_tokens.transpose(1, 2)) / (d ** 0.5)
            a = a.masked_fill(valid_pad.unsqueeze(1), float("-inf"))
            a = F.softmax(a, dim=-1)
            attn_pool = torch.bmm(a, valid_tokens).squeeze(1)

            combined = torch.cat([attn_pool, mean_pool, max_pool, count_norm], dim=-1)
            pooled[valid_idx] = self.norm(self.combine(combined)).to(dtype)

        return pooled, slot_tokens


class SlotXYDecoderLayer(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, d_ff=D_FF, dropout=DROPOUT):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True)
        self.self_norm = nn.LayerNorm(d_model)
        self.self_ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout))
        self.self_ffn_norm = nn.LayerNorm(d_model)

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True)
        self.cross_norm = nn.LayerNorm(d_model)
        self.cross_ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout))
        self.cross_ffn_norm = nn.LayerNorm(d_model)

    def forward(self, q, patch_features, slot_pad):
        sa_out, _ = self.self_attn(q, q, q, key_padding_mask=slot_pad)
        q = self.self_norm(q + sa_out)
        q = self.self_ffn_norm(q + self.self_ffn(q))

        ca_out, _ = self.cross_attn(q, patch_features, patch_features)
        q = self.cross_norm(q + ca_out)
        q = self.cross_ffn_norm(q + self.cross_ffn(q))
        return q


class SlotXYDecoder(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, d_ff=D_FF, dropout=DROPOUT,
                 n_layers=N_LAYERS_DECODER):
        super().__init__()
        self.query_proj = nn.Sequential(
            nn.Linear(d_model * 2, d_model), nn.GELU(), nn.Linear(d_model, d_model))
        self.query_norm = nn.LayerNorm(d_model)

        self.layers = nn.ModuleList([
            SlotXYDecoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

        self.x_head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, GRID_W))
        self.y_head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, GRID_H))

    def forward(self, patch_features, slot_query_input, slot_mask):
        q = self.query_norm(self.query_proj(slot_query_input))
        slot_pad = ~slot_mask

        for layer in self.layers:
            q = ckpt_fn(layer, q, patch_features, slot_pad, use_reentrant=False)

        return self.x_head(q), self.y_head(q)


class ViTPatchMaskModel(nn.Module):
    def __init__(self, n_channels, d_model=D_MODEL, n_heads=N_HEADS,
                 n_layers_temporal=N_LAYERS_TEMPORAL,
                 n_layers_spatial=N_LAYERS_SPATIAL,
                 d_ff=D_FF, dropout=DROPOUT):
        super().__init__()

        self.intra_frame = IntraFrameModule(d_model, n_heads, d_ff, dropout)
        self.channel_emb = nn.Embedding(n_channels, d_model)
        self.time_proj = nn.Sequential(
            nn.Linear(TIME_DIM, 64),
            nn.GELU(),
            nn.Linear(64, d_model),
        )
        self.temporal_norm = nn.LayerNorm(d_model)

        temporal_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.temporal_transformer = nn.TransformerEncoder(
            temporal_layer, num_layers=n_layers_temporal,
            enable_nested_tensor=False,
        )

        self.patch_pos_emb = nn.Parameter(
            torch.randn(1, N_PATCHES, d_model) * 0.02
        )
        self.spatial_norm = nn.LayerNorm(d_model)

        spatial_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.spatial_transformer = nn.TransformerEncoder(
            spatial_layer, num_layers=n_layers_spatial,
            enable_nested_tensor=False,
        )

        self.slot_decoder = SlotXYDecoder(d_model, n_heads, d_ff, dropout)

    def forward(self, channel_ids, telop_feats, telop_mask, time_feats, frame_mask):
        B, T, K, _ = telop_feats.shape
        device = telop_feats.device
        dtype = telop_feats.dtype
        BT = B * T
        D = D_MODEL

        slot_flat = telop_feats.reshape(BT, K, SLOT_DIM)
        smask_flat = telop_mask.reshape(BT, K)

        pooled_list, slot_tok_list = [], []
        for start in range(0, BT, INTRA_CHUNK):
            end = min(start + INTRA_CHUNK, BT)
            p, s = self.intra_frame(slot_flat[start:end], smask_flat[start:end])
            pooled_list.append(p)
            slot_tok_list.append(s)

        frame_tokens = torch.cat(pooled_list, dim=0).reshape(B, T, D)
        all_slot_tokens = torch.cat(slot_tok_list, dim=0).reshape(B, T, K, D)

        ch = self.channel_emb(channel_ids).unsqueeze(1).expand(-1, T, -1)
        time = self.time_proj(time_feats)
        tokens = self.temporal_norm(frame_tokens + ch + time)

        temporal_out = self.temporal_transformer(
            tokens, src_key_padding_mask=~frame_mask
        )

        temporal_flat = temporal_out.reshape(BT, D)
        slot_tok_flat = all_slot_tokens.reshape(BT, K, D)
        valid_flat = frame_mask.reshape(BT) & smask_flat.any(dim=1)
        valid_idx = valid_flat.nonzero(as_tuple=True)[0]
        n_valid = valid_idx.shape[0]

        all_x = torch.zeros(BT, K, GRID_W, device=device, dtype=dtype)
        all_y = torch.zeros(BT, K, GRID_H, device=device, dtype=dtype)

        if n_valid > 0:
            valid_ctx = temporal_flat[valid_idx]
            valid_slot = slot_tok_flat[valid_idx]
            valid_smask = smask_flat[valid_idx]

            queries = valid_ctx.unsqueeze(1) + self.patch_pos_emb
            queries = self.spatial_norm(queries)

            patch_feat_list = []
            for start in range(0, n_valid, SPATIAL_CHUNK):
                end = min(start + SPATIAL_CHUNK, n_valid)
                spatial_out = self.spatial_transformer(queries[start:end])
                patch_feat_list.append(spatial_out)

            patch_features = torch.cat(patch_feat_list, dim=0)

            ctx_expand = valid_ctx.unsqueeze(1).expand(-1, K, -1)
            slot_query_input = torch.cat([ctx_expand, valid_slot], dim=-1)

            x_list, y_list = [], []
            for start in range(0, n_valid, SLOT_DECODE_CHUNK):
                end = min(start + SLOT_DECODE_CHUNK, n_valid)
                xo, yo = self.slot_decoder(
                    patch_features[start:end],
                    slot_query_input[start:end],
                    valid_smask[start:end],
                )
                x_list.append(xo)
                y_list.append(yo)

            all_x[valid_idx] = torch.cat(x_list, dim=0).to(dtype)
            all_y[valid_idx] = torch.cat(y_list, dim=0).to(dtype)

        return all_x.reshape(B, T, K, GRID_W), all_y.reshape(B, T, K, GRID_H)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ViTPatchMaskModel(n_channels=len(channels)).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"🖥️  Device: {device}")
print(f"📐 파라미터: {n_params:,}")
print(f"   intra slot self-attn: {N_LAYERS_INTRA_SLOT}L (pre-norm, slot_pos_emb 내장)")
print(f"   temporal: {N_LAYERS_TEMPORAL}L  spatial: {N_LAYERS_SPATIAL}L")
print(f"   decoder: {N_LAYERS_DECODER}L (self-attn + cross-attn × {N_LAYERS_DECODER})")
print(f"   gradient checkpointing: ✅ (decoder)")
print(f"   d_model={D_MODEL}  n_heads={N_HEADS}  d_ff={D_FF}")
print(f"   INTRA_CHUNK={INTRA_CHUNK}  SLOT_DECODE_CHUNK={SLOT_DECODE_CHUNK}")
print(f"   출력: x[{GRID_W}] + y[{GRID_H}] → 외적으로 {GRID_H}×{GRID_W} 복원")

/tmp/ipykernel_2350794/3819507921.py:35: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.slot_self_attn = nn.TransformerEncoder(


🖥️  Device: cuda
📐 파라미터: 12,396,704
   intra slot self-attn: 2L (pre-norm, slot_pos_emb 내장)
   temporal: 4L  spatial: 4L
   decoder: 6L (self-attn + cross-attn × 6)
   gradient checkpointing: ✅ (decoder)
   d_model=256  n_heads=8  d_ff=512
   INTRA_CHUNK=512  SLOT_DECODE_CHUNK=512
   출력: x[80] + y[80] → 외적으로 80×80 복원


In [ ]:
# %% 셀 4: 학습 (Slot XY 분리 예측 + 축별 Focal Loss)
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 50
LR = 1e-4
FOCAL_ALPHA_X = 2.7 / (1 + 2.7)    # ≈ 0.730
FOCAL_ALPHA_Y = 25.6 / (1 + 25.6)  # ≈ 0.962
FOCAL_GAMMA = 2.0
SAVE_DIR = "./model/8_layout_slot_xy"
os.makedirs(SAVE_DIR, exist_ok=True)

_x_range = torch.arange(GRID_W, device=device).unsqueeze(0)
_y_range = torch.arange(GRID_H, device=device).unsqueeze(0)


def bbox_to_gt_xy(slot_bbox, active_mask):
    bbox = slot_bbox[active_mask].long()
    x0, y0, x1, y1 = bbox[:, 0], bbox[:, 1], bbox[:, 2], bbox[:, 3]
    gt_x = ((_x_range >= x0.unsqueeze(1)) & (_x_range < x1.unsqueeze(1))).float()
    gt_y = ((_y_range >= y0.unsqueeze(1)) & (_y_range < y1.unsqueeze(1))).float()
    return gt_x, gt_y


def focal_loss_1d(logits, targets, alpha, gamma=FOCAL_GAMMA):
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    probs = torch.sigmoid(logits)
    p_t = probs * targets + (1 - probs) * (1 - targets)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    focal_weight = alpha_t * (1 - p_t) ** gamma
    return (focal_weight * bce).mean()


def slot_xy_loss(x_logits, y_logits, slot_bbox, active_mask):
    if not active_mask.any():
        return x_logits.sum() * 0.0

    active_x = x_logits[active_mask]
    active_y = y_logits[active_mask]
    gt_x, gt_y = bbox_to_gt_xy(slot_bbox, active_mask)

    loss_x = focal_loss_1d(active_x, gt_x, alpha=FOCAL_ALPHA_X)
    loss_y = focal_loss_1d(active_y, gt_y, alpha=FOCAL_ALPHA_Y)

    return loss_x + loss_y


@torch.no_grad()
def compute_slot_metrics(x_logits, y_logits, slot_bbox, active_mask,
                          thresholds=(0.2, 0.3, 0.4, 0.5, 0.6, 0.7)):
    if not active_mask.any():
        return None

    active_x = torch.sigmoid(x_logits[active_mask])
    active_y = torch.sigmoid(y_logits[active_mask])
    gt_x, gt_y = bbox_to_gt_xy(slot_bbox, active_mask)

    # ── 1) index-aligned slot F1 (2D) ──
    pred_2d = active_y.unsqueeze(-1) * active_x.unsqueeze(-2)
    gt_2d   = gt_y.unsqueeze(-1) * gt_x.unsqueeze(-2)
    gt_bool = gt_2d.bool()

    pred_flat = pred_2d.reshape(-1)
    gt_flat   = gt_bool.reshape(-1)

    best_f1, best_th = 0.0, 0.5
    result_05 = {}

    for th in thresholds:
        pred_bin = pred_flat > th
        tp = (pred_bin & gt_flat).sum().item()
        fp = (pred_bin & ~gt_flat).sum().item()
        fn = (~pred_bin & gt_flat).sum().item()
        p = tp / (tp + fp + 1e-8)
        r = tp / (tp + fn + 1e-8)
        f1 = 2 * p * r / (p + r + 1e-8)
        if th == 0.5:
            result_05 = {"P": p, "R": r, "F1": f1}
        if f1 > best_f1:
            best_f1 = f1
            best_th = th

    result_05["bestF1"] = best_f1
    result_05["bestTh"] = best_th

    # ── 2) x-only / y-only F1 ──
    gt_x_bool = gt_x.bool()
    gt_y_bool = gt_y.bool()

    best_xf1 = 0.0
    best_yf1 = 0.0
    for th in thresholds:
        # x-only
        xb = active_x.reshape(-1) > th
        xg = gt_x_bool.reshape(-1)
        tp = (xb & xg).sum().item()
        fp = (xb & ~xg).sum().item()
        fn = (~xb & xg).sum().item()
        p = tp / (tp + fp + 1e-8)
        r = tp / (tp + fn + 1e-8)
        xf1 = 2 * p * r / (p + r + 1e-8)
        if xf1 > best_xf1:
            best_xf1 = xf1

        # y-only
        yb = active_y.reshape(-1) > th
        yg = gt_y_bool.reshape(-1)
        tp = (yb & yg).sum().item()
        fp = (yb & ~yg).sum().item()
        fn = (~yb & yg).sum().item()
        p = tp / (tp + fp + 1e-8)
        r = tp / (tp + fn + 1e-8)
        yf1 = 2 * p * r / (p + r + 1e-8)
        if yf1 > best_yf1:
            best_yf1 = yf1

    result_05["xF1"] = best_xf1
    result_05["yF1"] = best_yf1

    # ── 3) Hungarian-matched slot F1 ──
    B, T, K = active_mask.shape
    from scipy.optimize import linear_sum_assignment

    x_prob = torch.sigmoid(x_logits)
    y_prob = torch.sigmoid(y_logits)

    has_frame = active_mask.any(dim=2)
    b_idx, t_idx = has_frame.nonzero(as_tuple=True)

    n_total = len(b_idx)
    if n_total > 500:
        sample_idx = torch.randperm(n_total, device=x_logits.device)[:500]
        b_idx = b_idx[sample_idx]
        t_idx = t_idx[sample_idx]

    hung_tp, hung_fp, hung_fn = 0, 0, 0
    best_hung_th = best_th  # aligned와 같은 threshold 사용

    for bi, ti in zip(b_idx.tolist(), t_idx.tolist()):
        am = active_mask[bi, ti]
        active_slots = am.nonzero(as_tuple=True)[0]
        n_active = len(active_slots)
        if n_active == 0:
            continue

        # 각 활성 슬롯의 pred 2D mask
        xp = x_prob[bi, ti, active_slots]  # (n_active, W)
        yp = y_prob[bi, ti, active_slots]  # (n_active, H)
        bb = slot_bbox[bi, ti, active_slots].long()  # (n_active, 4)

        # pred binary
        pred_masks = (yp.unsqueeze(-1) * xp.unsqueeze(-2)) > best_hung_th  # (n_active, H, W)

        # gt binary
        gt_masks = torch.zeros(n_active, GRID_H, GRID_W, device=x_logits.device, dtype=torch.bool)
        for j in range(n_active):
            gt_masks[j, bb[j, 1]:bb[j, 3], bb[j, 0]:bb[j, 2]] = True

        # IoU 행렬 (n_active × n_active)
        inter = (pred_masks.unsqueeze(1) & gt_masks.unsqueeze(0)).sum(dim=(-1, -2)).float()
        union = (pred_masks.unsqueeze(1) | gt_masks.unsqueeze(0)).sum(dim=(-1, -2)).float()
        iou_matrix = inter / (union + 1e-8)

        # Hungarian matching (cost = 1 - IoU)
        cost = (1 - iou_matrix).cpu().numpy()
        row_ind, col_ind = linear_sum_assignment(cost)

        # matched pairs로 F1 계산
        for ri, ci in zip(row_ind, col_ind):
            pm = pred_masks[ri]
            gm = gt_masks[ci]
            hung_tp += (pm & gm).sum().item()
            hung_fp += (pm & ~gm).sum().item()
            hung_fn += (~pm & gm).sum().item()

    hung_p = hung_tp / (hung_tp + hung_fp + 1e-8)
    hung_r = hung_tp / (hung_tp + hung_fn + 1e-8)
    hung_f1 = 2 * hung_p * hung_r / (hung_p + hung_r + 1e-8)
    result_05["hungF1"] = hung_f1

    # ── 4) merged frame F1 ──
    amask = active_mask.unsqueeze(-1)
    x_prob_m = x_prob * amask.float()
    y_prob_m = y_prob * amask.float()

    b_idx2, t_idx2 = has_frame.nonzero(as_tuple=True)
    n_total2 = len(b_idx2)
    if n_total2 > 2000:
        sample_idx = torch.randperm(n_total2, device=x_logits.device)[:2000]
        b_idx2 = b_idx2[sample_idx]
        t_idx2 = t_idx2[sample_idx]

    merged_preds = []
    merged_gts   = []
    for bi, ti in zip(b_idx2.tolist(), t_idx2.tolist()):
        xp = x_prob_m[bi, ti]
        yp = y_prob_m[bi, ti]
        am = active_mask[bi, ti]
        bb = slot_bbox[bi, ti]

        slot_2d = yp[am].unsqueeze(-1) * xp[am].unsqueeze(-2)
        pred_merged = slot_2d.max(dim=0).values

        gt_boxes = bb[am].long()
        gt_merged = torch.zeros(GRID_H, GRID_W, device=x_logits.device)
        for box in gt_boxes:
            gt_merged[box[1]:box[3], box[0]:box[2]] = 1.0

        merged_preds.append(pred_merged)
        merged_gts.append(gt_merged)

    mp = torch.stack(merged_preds).reshape(-1)
    mg = torch.stack(merged_gts).reshape(-1).bool()

    best_mf1, best_mth = 0.0, 0.5
    for th in thresholds:
        pred_bin = mp > th
        tp = (pred_bin & mg).sum().item()
        fp = (pred_bin & ~mg).sum().item()
        fn = (~pred_bin & mg).sum().item()
        p = tp / (tp + fp + 1e-8)
        r = tp / (tp + fn + 1e-8)
        f1 = 2 * p * r / (p + r + 1e-8)
        if f1 > best_mf1:
            best_mf1 = f1
            best_mth = th

    result_05["mBestF1"] = best_mf1
    result_05["mBestTh"] = best_mth

    return result_05

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_eval_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum, train_batches = 0.0, 0

    for batch in tqdm(train_loader, desc=f"[{epoch}/{EPOCHS}] train", leave=False):
        channel_ids = batch["channel_ids"].to(device)
        telop_feats = batch["telop_feats"].to(device)
        telop_mask  = batch["telop_mask"].to(device)
        time_feats  = batch["time_feats"].to(device)
        slot_bbox   = batch["slot_bbox"].to(device)
        frame_mask  = batch["frame_mask"].to(device)

        active_mask = telop_mask & frame_mask.unsqueeze(-1)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            x_logits, y_logits = model(
                channel_ids, telop_feats, telop_mask, time_feats, frame_mask
            )
            loss = slot_xy_loss(x_logits, y_logits, slot_bbox, active_mask)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss_sum += loss.item()
        train_batches += 1

    model.eval()
    eval_loss_sum, eval_batches = 0.0, 0
    all_metrics = []

    with torch.no_grad():
        for batch in eval_loader:
            channel_ids = batch["channel_ids"].to(device)
            telop_feats = batch["telop_feats"].to(device)
            telop_mask  = batch["telop_mask"].to(device)
            time_feats  = batch["time_feats"].to(device)
            slot_bbox   = batch["slot_bbox"].to(device)
            frame_mask  = batch["frame_mask"].to(device)

            active_mask = telop_mask & frame_mask.unsqueeze(-1)

            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                x_logits, y_logits = model(
                    channel_ids, telop_feats, telop_mask, time_feats, frame_mask
                )
                loss = slot_xy_loss(x_logits, y_logits, slot_bbox, active_mask)

            eval_loss_sum += loss.item()
            eval_batches += 1

            m = compute_slot_metrics(x_logits, y_logits, slot_bbox, active_mask)
            if m is not None:
                all_metrics.append(m)

    scheduler.step()

    train_loss = train_loss_sum / max(train_batches, 1)
    eval_loss  = eval_loss_sum  / max(eval_batches, 1)
    lr_now = optimizer.param_groups[0]["lr"]

    if all_metrics:
        avg_m = {k: np.mean([m[k] for m in all_metrics])
                 for k in all_metrics[0] if "Th" not in k}
        avg_sth = np.mean([m["bestTh"] for m in all_metrics])
    else:
        avg_m = {"P": 0, "R": 0, "F1": 0, "bestF1": 0, "mBestF1": 0}
        avg_sth = 0.5

    print(
        f"[{epoch:>3}/{EPOCHS}]  "
        f"train={train_loss:.4f}  eval={eval_loss:.4f}  "
        f"P={avg_m.get('P',0):.3f}  R={avg_m.get('R',0):.3f}  "
        f"F1={avg_m.get('F1',0):.3f}  best={avg_m.get('bestF1',0):.3f}@{avg_sth:.2f}  "
        f"hF1={avg_m.get('hungF1',0):.3f}  "
        f"mF1={avg_m.get('mBestF1',0):.3f}  "
        f"xF1={avg_m.get('xF1',0):.3f}  yF1={avg_m.get('yF1',0):.3f}  "
        f"lr={lr_now:.2e}"
    )
    
    ckpt = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "eval_loss": eval_loss,
        "eval_metrics": avg_m,
        "channel2id": channel2id,
    }
    torch.save(ckpt, os.path.join(SAVE_DIR, f"epoch_{epoch:03d}.pt"))

    if eval_loss < best_eval_loss:
        best_eval_loss = eval_loss
        torch.save(ckpt, os.path.join(SAVE_DIR, "best.pt"))
        print(f"   💾 best 갱신 (eval_loss={eval_loss:.4f})")

print(f"\n✅ 완료. Best eval_loss={best_eval_loss:.4f}")

[  1/50]  train=0.0729  eval=0.0692  sF1=0.161  hF1=0.162  mF1=0.395  xF1=0.723  yF1=0.212  lr=9.99e-05
   💾 best 갱신 (eval_loss=0.0692)


[2/50] train:  12%|█▏        | 81/696 [01:36<10:51,  1.06s/it]

                                                               
[  1/50]  train=0.0729  eval=0.0698  P=0.017  R=0.000  F1=0.000  best=0.162@0.38  merged=0.403  lr=9.99e-05
   💾 best 갱신 (eval_loss=0.0698)
                                                               
[  2/50]  train=0.0682  eval=0.0634  P=0.246  R=0.083  F1=0.105  best=0.197@0.41  merged=0.463  lr=9.96e-05
   💾 best 갱신 (eval_loss=0.0634)
                                                               
[  3/50]  train=0.0636  eval=0.0612  P=0.242  R=0.170  F1=0.160  best=0.220@0.44  merged=0.485  lr=9.91e-05
   💾 best 갱신 (eval_loss=0.0612)
                                                               
[  4/50]  train=0.0616  eval=0.0594  P=0.220  R=0.218  F1=0.186  best=0.217@0.45  merged=0.472  lr=9.84e-05
   💾 best 갱신 (eval_loss=0.0594)
                                                               
[  5/50]  train=0.0602  eval=0.0582  P=0.240  R=0.196  F1=0.173  best=0.234@0.43  merged=0.497  lr=9.76e-05
   💾 best 갱신 (eval_loss=0.0582)
                                                               
[  6/50]  train=0.0590  eval=0.0583  P=0.277  R=0.171  F1=0.178  best=0.237@0.43  merged=0.484  lr=9.65e-05
                                                               
[  7/50]  train=0.0583  eval=0.0563  P=0.237  R=0.244  F1=0.211  best=0.241@0.46  merged=0.488  lr=9.52e-05
   💾 best 갱신 (eval_loss=0.0563)
                                                               
[  8/50]  train=0.0572  eval=0.0553  P=0.252  R=0.260  F1=0.215  best=0.254@0.46  merged=0.508  lr=9.38e-05
   💾 best 갱신 (eval_loss=0.0553)
                                                               
[  9/50]  train=0.0566  eval=0.0552  P=0.244  R=0.261  F1=0.222  best=0.258@0.46  merged=0.511  lr=9.22e-05
   💾 best 갱신 (eval_loss=0.0552)
                                                                
[ 10/50]  train=0.0555  eval=0.0538  P=0.260  R=0.336  F1=0.260  best=0.281@0.49  merged=0.525  lr=9.05e-05
   💾 best 갱신 (eval_loss=0.0538)
                                                                
[ 11/50]  train=0.0548  eval=0.0538  P=0.259  R=0.283  F1=0.246  best=0.267@0.47  merged=0.517  lr=8.85e-05
                                                                
[ 12/50]  train=0.0542  eval=0.0533  P=0.259  R=0.311  F1=0.250  best=0.274@0.46  merged=0.511  lr=8.64e-05
   💾 best 갱신 (eval_loss=0.0533)
                                                                
[ 13/50]  train=0.0543  eval=0.0543  P=0.247  R=0.318  F1=0.254  best=0.284@0.48  merged=0.522  lr=8.42e-05
                                                                
[ 14/50]  train=0.0534  eval=0.0533  P=0.264  R=0.295  F1=0.247  best=0.276@0.47  merged=0.524  lr=8.19e-05
                                                                
[ 15/50]  train=0.0530  eval=0.0527  P=0.260  R=0.309  F1=0.252  best=0.284@0.47  merged=0.526  lr=7.94e-05
   💾 best 갱신 (eval_loss=0.0527)
                                                                
[ 16/50]  train=0.0522  eval=0.0523  P=0.261  R=0.370  F1=0.280  best=0.301@0.49  merged=0.546  lr=7.68e-05
   💾 best 갱신 (eval_loss=0.0523)
                                                                
[ 17/50]  train=0.0519  eval=0.0518  P=0.266  R=0.334  F1=0.269  best=0.289@0.47  merged=0.534  lr=7.41e-05
   💾 best 갱신 (eval_loss=0.0518)
                                                                
[ 18/50]  train=0.0510  eval=0.0513  P=0.254  R=0.343  F1=0.265  best=0.294@0.48  merged=0.545  lr=7.13e-05
   💾 best 갱신 (eval_loss=0.0513)
                                                                
[ 19/50]  train=0.0505  eval=0.0517  P=0.296  R=0.312  F1=0.261  best=0.284@0.47  merged=0.527  lr=6.84e-05
                                                                
[ 20/50]  train=0.0500  eval=0.0509  P=0.260  R=0.376  F1=0.282  best=0.306@0.49  merged=0.545  lr=6.55e-05
   💾 best 갱신 (eval_loss=0.0509)
                                                                
[ 21/50]  train=0.0496  eval=0.0499  P=0.269  R=0.354  F1=0.278  best=0.306@0.47  merged=0.551  lr=6.24e-05
   💾 best 갱신 (eval_loss=0.0499)
                                                                
[ 22/50]  train=0.0496  eval=0.0505  P=0.273  R=0.339  F1=0.277  best=0.305@0.48  merged=0.548  lr=5.94e-05
                                                                
[ 23/50]  train=0.0484  eval=0.0494  P=0.253  R=0.424  F1=0.302  best=0.319@0.50  merged=0.562  lr=5.63e-05
   💾 best 갱신 (eval_loss=0.0494)
                                                                
[ 24/50]  train=0.0478  eval=0.0497  P=0.313  R=0.363  F1=0.289  best=0.315@0.49  merged=0.553  lr=5.31e-05
                                                                
[ 25/50]  train=0.0472  eval=0.0496  P=0.279  R=0.415  F1=0.307  best=0.325@0.50  merged=0.566  lr=5.00e-05
                                                                
[ 26/50]  train=0.0468  eval=0.0494  P=0.294  R=0.416  F1=0.312  best=0.330@0.51  merged=0.567  lr=4.69e-05
                                                                
[ 27/50]  train=0.0463  eval=0.0488  P=0.299  R=0.424  F1=0.319  best=0.334@0.51  merged=0.574  lr=4.37e-05
   💾 best 갱신 (eval_loss=0.0488)
                                                                
[ 28/50]  train=0.0459  eval=0.0503  P=0.291  R=0.409  F1=0.307  best=0.327@0.49  merged=0.565  lr=4.06e-05
                                                                
[ 29/50]  train=0.0455  eval=0.0491  P=0.306  R=0.394  F1=0.313  best=0.327@0.50  merged=0.565  lr=3.76e-05
                                                                
[ 30/50]  train=0.0450  eval=0.0489  P=0.303  R=0.425  F1=0.321  best=0.339@0.50  merged=0.574  lr=3.45e-05
                                                                
[ 31/50]  train=0.0439  eval=0.0491  P=0.277  R=0.458  F1=0.326  best=0.345@0.51  merged=0.579  lr=3.16e-05
                                                                
[ 32/50]  train=0.0436  eval=0.0489  P=0.281  R=0.443  F1=0.319  best=0.336@0.51  merged=0.577  lr=2.87e-05
                                                                
[ 33/50]  train=0.0428  eval=0.0485  P=0.284  R=0.447  F1=0.324  best=0.338@0.52  merged=0.575  lr=2.59e-05
   💾 best 갱신 (eval_loss=0.0485)
                                                                
[ 34/50]  train=0.0427  eval=0.0488  P=0.275  R=0.466  F1=0.329  best=0.345@0.52  merged=0.582  lr=2.32e-05
                                                                
[ 35/50]  train=0.0421  eval=0.0497  P=0.278  R=0.467  F1=0.333  best=0.349@0.52  merged=0.585  lr=2.06e-05
                                                                
[ 36/50]  train=0.0417  eval=0.0490  P=0.281  R=0.456  F1=0.330  best=0.350@0.52  merged=0.585  lr=1.81e-05
                                                                
[ 37/50]  train=0.0412  eval=0.0485  P=0.278  R=0.460  F1=0.331  best=0.347@0.52  merged=0.583  lr=1.58e-05
   💾 best 갱신 (eval_loss=0.0485)
                                                                
[ 38/50]  train=0.0408  eval=0.0486  P=0.274  R=0.462  F1=0.328  best=0.348@0.53  merged=0.582  lr=1.36e-05
                                                                
[ 39/50]  train=0.0406  eval=0.0491  P=0.278  R=0.461  F1=0.330  best=0.350@0.52  merged=0.584  lr=1.15e-05
                                                                
[ 40/50]  train=0.0399  eval=0.0488  P=0.275  R=0.474  F1=0.333  best=0.352@0.52  merged=0.586  lr=9.55e-06
                                                                
[ 41/50]  train=0.0400  eval=0.0492  P=0.281  R=0.474  F1=0.336  best=0.355@0.52  merged=0.590  lr=7.78e-06
                                                                
[ 42/50]  train=0.0396  eval=0.0492  P=0.276  R=0.471  F1=0.333  best=0.352@0.52  merged=0.588  lr=6.18e-06
                                                                
[ 43/50]  train=0.0390  eval=0.0493  P=0.275  R=0.473  F1=0.333  best=0.352@0.52  merged=0.589  lr=4.76e-06
                                                                
[ 44/50]  train=0.0389  eval=0.0496  P=0.277  R=0.468  F1=0.333  best=0.351@0.52  merged=0.589  lr=3.51e-06
                                                                
[ 45/50]  train=0.0391  eval=0.0494  P=0.274  R=0.474  F1=0.333  best=0.353@0.53  merged=0.588  lr=2.45e-06
                                                                
[ 46/50]  train=0.0389  eval=0.0494  P=0.275  R=0.475  F1=0.334  best=0.353@0.53  merged=0.589  lr=1.57e-06
                                                                
[ 47/50]  train=0.0387  eval=0.0495  P=0.275  R=0.476  F1=0.334  best=0.354@0.53  merged=0.589  lr=8.86e-07
                                                                
[ 48/50]  train=0.0387  eval=0.0495  P=0.275  R=0.477  F1=0.334  best=0.354@0.53  merged=0.590  lr=3.94e-07
                                                                
[ 49/50]  train=0.0385  eval=0.0495  P=0.275  R=0.476  F1=0.334  best=0.354@0.53  merged=0.590  lr=9.87e-08
                                                                
[ 50/50]  train=0.0385  eval=0.0496  P=0.275  R=0.476  F1=0.334  best=0.354@0.53  merged=0.589  lr=0.00e+00

✅ 완료. Best eval_loss=0.0485